# Bangla Hallucination Detection — v4.0

**Strategy: Feature Engineering + Pretrained Signals + Light Classifier**

No fine-tuning on 299 samples. Instead:
1. NLI entailment/contradiction scores (mDeBERTa zero-shot)
2. Multilingual sentence embedding similarity (paraphrase-multilingual-MiniLM-L12-v2)
3. Hand-crafted features (token overlap, length ratios, number extraction, etc.)
4. Light classifier (Logistic Regression + XGBoost) on stacked features
5. 5-fold CV with proper OOF evaluation

In [ ]:
!pip install -q transformers sentence-transformers xgboost
!pip install -q git+https://github.com/csebuetnlp/normalizer

In [ ]:
import json, gc, os, re, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from normalizer import normalize
from transformers import pipeline, AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SKPipeline
import xgboost as xgb
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
DATA_PATH = "/kaggle/input/competitions/bengali-hallucination/dataset samples.json"
TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
SUBMISSION_PATH = "/kaggle/working/submission_v4.0.csv"

NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
EMB_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Load & Clean Data

In [ ]:
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../bengali-hallucination/dataset samples.json"
    TEST_PATH = "../bengali-hallucination/test set.csv"

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)
df = pd.DataFrame(records)
print(f"Train: {len(df)}")

NO_CTX = {"", "nan", "NaN", "[NULL]", None}
def clean(text):
    if pd.isna(text) or str(text).strip() in NO_CTX:
        return ""
    return normalize(str(text).strip())

for col in ['context', 'prompt_bn', 'response_bn']:
    df[col] = df[col].apply(clean)

df['has_context'] = df['context'].str.len() > 0
print(f"With context: {df['has_context'].sum()}, Without: {(~df['has_context']).sum()}")
print(f"Label 0: {(df['label']==0).sum()}, Label 1: {(df['label']==1).sum()}")

## 2. Hand-Crafted Feature Engineering

In [ ]:
# Bengali numeral to int mapping
BN_DIGITS = {'০':0,'১':1,'২':2,'৩':3,'৪':4,'৫':5,'৬':6,'৭':7,'৮':8,'৯':9}

def bn_to_int(s):
    """Convert Bengali numeral string to int, return None on failure."""
    try:
        return int(s)
    except:
        pass
    try:
        s2 = s
        for bn, ar in BN_DIGITS.items():
            s2 = s2.replace(bn, str(ar))
        return int(s2)
    except:
        return None

def extract_numbers(text):
    """Extract all numbers (Bengali + Arabic) from text."""
    nums = re.findall(r'[০-৯]+|[0-9]+(?:\.[0-9]+)?', text)
    result = []
    for n in nums:
        v = bn_to_int(n)
        if v is not None:
            result.append(v)
    return result

def tokenize_bangla(text):
    """Simple whitespace + punctuation tokenizer for Bangla."""
    return [w for w in re.split(r'[\s,\.\?\!\?\"\'\:\;\(\)\[\]\{\}]+', text) if len(w) > 0]

def jaccard(tokens_a, tokens_b):
    if not tokens_a or not tokens_b:
        return 0.0
    set_a, set_b = set(tokens_a), set(tokens_b)
    return len(set_a & set_b) / max(len(set_a | set_b), 1)

def common_ratio(tokens_a, tokens_b):
    if not tokens_b:
        return 0.0
    return len(set(tokens_a) & set(tokens_b)) / max(len(set(tokens_b)), 1)

def build_features(data_df):
    """Build hand-crafted feature matrix."""
    feats = pd.DataFrame(index=data_df.index)

    # Basic length features
    feats['ctx_len'] = data_df['context'].str.len()
    feats['prompt_len'] = data_df['prompt_bn'].str.len()
    feats['resp_len'] = data_df['response_bn'].str.len()
    feats['has_context'] = data_df['has_context'].astype(int)

    # Length ratios
    feats['resp_prompt_ratio'] = feats['resp_len'] / (feats['prompt_len'] + 1)
    feats['resp_ctx_ratio'] = feats['resp_len'] / (feats['ctx_len'] + 1)

    # Token-level features
    prompt_toks = data_df['prompt_bn'].apply(tokenize_bangla)
    resp_toks = data_df['response_bn'].apply(tokenize_bangla)
    ctx_toks = data_df['context'].apply(tokenize_bangla)

    feats['prompt_n_tokens'] = prompt_toks.apply(len)
    feats['resp_n_tokens'] = resp_toks.apply(len)
    feats['ctx_n_tokens'] = ctx_toks.apply(len)

    # Overlap features: how much of the response appears in context
    feats['resp_in_ctx_jaccard'] = [
        jaccard(r, c) for r, c in zip(resp_toks, ctx_toks)
    ]
    feats['resp_in_ctx_ratio'] = [
        common_ratio(r, c) for r, c in zip(resp_toks, ctx_toks)
    ]
    feats['ctx_in_resp_ratio'] = [
        common_ratio(c, r) for r, c in zip(resp_toks, ctx_toks)
    ]

    # Prompt-response overlap (answer should relate to question)
    feats['resp_prompt_jaccard'] = [
        jaccard(r, p) for r, p in zip(resp_toks, prompt_toks)
    ]
    feats['resp_in_prompt_ratio'] = [
        common_ratio(r, p) for r, p in zip(resp_toks, prompt_toks)
    ]

    # Number extraction features
    ctx_nums = data_df['context'].apply(extract_numbers)
    resp_nums = data_df['response_bn'].apply(extract_numbers)
    prompt_nums = data_df['prompt_bn'].apply(extract_numbers)

    feats['resp_has_number'] = resp_nums.apply(len).clip(upper=1)
    feats['ctx_has_number'] = ctx_nums.apply(len).clip(upper=1)
    feats['prompt_has_number'] = prompt_nums.apply(len).clip(upper=1)

    # Check if any response number appears in context
    def numbers_match(rn, cn):
        if not rn:
            return 0
        return int(any(n in cn for n in rn))

    feats['resp_num_in_ctx'] = [numbers_match(rn, cn) for rn, cn in zip(resp_nums, ctx_nums)]

    # Response is very short (likely a name/entity)
    feats['resp_very_short'] = (feats['resp_len'] < 15).astype(int)
    feats['resp_one_word'] = (feats['resp_n_tokens'] <= 2).astype(int)

    # Response starts with context (copy-paste hallucination)
    feats['resp_starts_with_ctx'] = [
        int(str(r).startswith(str(c)[:20])) if c and r else 0
        for r, c in zip(data_df['response_bn'], data_df['context'])
    ]

    return feats.fillna(0)

print("Building hand-crafted features...")
hc_train = build_features(df)
print(f"Hand-crafted features: {hc_train.shape[1]} columns")
hc_train.head()

## 3. NLI Entailment Scores

In [ ]:
print(f"Loading NLI model: {NLI_MODEL}")
nli_pipe = pipeline(
    'zero-shot-classification',
    model=NLI_MODEL,
    device=0 if device == 'cuda' else -1,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)
print("NLI ready.")

In [ ]:
def get_nli_scores(data_df):
    """Compute NLI entailment scores. Two strategies:
    - With context: premise=context, hypothesis=response
    - Without context: premise=prompt, hypothesis=response
    Also compute contradiction score."""
    entail_scores = np.full(len(data_df), 0.5)
    contra_scores = np.full(len(data_df), 0.5)

    for i, (_, row) in enumerate(tqdm(data_df.iterrows(), total=len(data_df), desc='NLI')):
        premise = row.context if row.context else row.prompt_bn
        if not premise:
            continue
        hypothesis = row.response_bn
        if not hypothesis:
            continue
        try:
            result = nli_pipe(
                premise,
                candidate_labels=[hypothesis],
                hypothesis_template='{}',
                multi_label=True,
            )
            entail_scores[i] = result['scores'][0]

            # For contradiction: swap premise/hypothesis and check entailment
            result2 = nli_pipe(
                hypothesis,
                candidate_labels=[premise],
                hypothesis_template='{}',
                multi_label=True,
            )
            contra_scores[i] = result2['scores'][0]
        except Exception as e:
            pass

    return entail_scores, contra_scores

print("Computing NLI scores for training data...")
nli_entail, nli_contra = get_nli_scores(df)
df['nli_entail'] = nli_entail
df['nli_contra'] = nli_contra
print(f"NLI entail: mean={nli_entail.mean():.3f}, std={nli_entail.std():.3f}")
print(f"NLI contra: mean={nli_contra.mean():.3f}, std={nli_contra.std():.3f}")

# Sanity check on context rows
ctx_mask = df['has_context']
ctx_acc = accuracy_score(df.loc[ctx_mask, 'label'], (nli_entail[ctx_mask] > 0.5).astype(int))
print(f"NLI accuracy on context rows: {ctx_acc:.4f}")

## 4. Sentence Embedding Similarity

In [ ]:
print(f"Loading embedding model: {EMB_MODEL}")
emb_model = SentenceTransformer(EMB_MODEL, device=str(device))
print("Embedding model ready.")

In [ ]:
def cosine_sim_matrix(texts_a, texts_b, batch_size=64):
    """Compute pairwise cosine similarity between two lists of texts."""
    emb_a = emb_model.encode(texts_a, batch_size=batch_size, show_progress_bar=True,
                             normalize_embeddings=True)
    emb_b = emb_model.encode(texts_b, batch_size=batch_size, show_progress_bar=True,
                             normalize_embeddings=True)
    # Since embeddings are normalized, dot product = cosine similarity
    sims = np.sum(emb_a * emb_b, axis=1)
    return sims, emb_a, emb_b

print("Encoding training texts...")
ctx_texts = df['context'].where(df['context'] != '', df['prompt_bn']).tolist()
resp_texts = df['response_bn'].tolist()
prompt_texts = df['prompt_bn'].tolist()

# Context/Prompt - Response similarity
sim_ctx_resp, emb_ctx, emb_resp = cosine_sim_matrix(ctx_texts, resp_texts)

# Prompt - Response similarity
sim_prompt_resp, _, _ = cosine_sim_matrix(prompt_texts, resp_texts)

df['sim_ctx_resp'] = sim_ctx_resp
df['sim_prompt_resp'] = sim_prompt_resp

print(f"sim_ctx_resp: mean={sim_ctx_resp.mean():.3f}, std={sim_ctx_resp.std():.3f}")
print(f"sim_prompt_resp: mean={sim_prompt_resp.mean():.3f}, std={sim_prompt_resp.std():.3f}")

# Check similarity alone on context rows
ctx_mask = df['has_context']
ctx_acc_sim = accuracy_score(df.loc[ctx_mask, 'label'], (sim_ctx_resp[ctx_mask] > 0.5).astype(int))
print(f"Embedding sim accuracy on context rows: {ctx_acc_sim:.4f}")

## 5. Assemble Full Feature Matrix

In [ ]:
# Combine all features
feature_cols = list(hc_train.columns) + ['nli_entail', 'nli_contra', 'sim_ctx_resp', 'sim_prompt_resp']

X_full = np.column_stack([
    hc_train.values,
    df['nli_entail'].values.reshape(-1, 1),
    df['nli_contra'].values.reshape(-1, 1),
    df['sim_ctx_resp'].values.reshape(-1, 1),
    df['sim_prompt_resp'].values.reshape(-1, 1),
])
y = df['label'].values

print(f"Feature matrix: {X_full.shape}")
print(f"Features: {feature_cols}")
print(f"Label distribution: 0={ (y==0).sum()}, 1={ (y==1).sum()}")

## 6. 5-Fold CV with Logistic Regression + XGBoost

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

# OOF predictions
oof_lr = np.zeros(len(df))
oof_xgb = np.zeros(len(df))

n0, n1 = (y == 0).sum(), (y == 1).sum()

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y)):
    print(f"\n--- Fold {fold} ---")
    X_tr, X_va = X_full[tr_idx], X_full[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # Logistic Regression with scaling
    lr = SKPipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            C=1.0, max_iter=1000, class_weight='balanced',
            random_state=RANDOM_STATE
        )),
    ])
    lr.fit(X_tr, y_tr)
    oof_lr[va_idx] = lr.predict_proba(X_va)[:, 1]

    # XGBoost
    xgb_clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
        scale_pos_weight=n0/n1, random_state=RANDOM_STATE,
        eval_metric='logloss', early_stopping_rounds=20,
        use_label_encoder=False, verbosity=0
    )
    xgb_clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[va_idx] = xgb_clf.predict_proba(X_va)[:, 1]

    lr_pred = (oof_lr[va_idx] >= 0.5).astype(int)
    xgb_pred = (oof_xgb[va_idx] >= 0.5).astype(int)
    print(f"  LR  Acc={accuracy_score(y_va, lr_pred):.4f}  F1={f1_score(y_va, lr_pred, average='macro'):.4f}")
    print(f"  XGB Acc={accuracy_score(y_va, xgb_pred):.4f}  F1={f1_score(y_va, xgb_pred, average='macro'):.4f}")

# Individual OOF scores
lr_p = (oof_lr >= 0.5).astype(int)
xgb_p = (oof_xgb >= 0.5).astype(int)
print(f"\n=== LR OOF:  Acc={accuracy_score(y, lr_p):.4f}  F1={f1_score(y, lr_p, average='macro'):.4f}")
print(f"=== XGB OOF: Acc={accuracy_score(y, xgb_p):.4f}  F1={f1_score(y, xgb_p, average='macro'):.4f}")
print(confusion_matrix(y, lr_p))
print(confusion_matrix(y, xgb_p))

## 7. Blend LR + XGBoost + NLI (Grid Search on OOF)

In [ ]:
nli_raw = df['nli_entail'].values
sim_raw = df['sim_ctx_resp'].values

# Grid search over 4-component blend
best_f1, best_w = 0, None
step = 0.05
for w_lr in np.arange(0, 1.01, step):
    for w_xgb in np.arange(0, 1.01 - w_lr, step):
        for w_nli in np.arange(0, 1.01 - w_lr - w_xgb, step):
            w_sim = 1.0 - w_lr - w_xgb - w_nli
            if w_sim < -0.001:
                continue
            blended = w_lr * oof_lr + w_xgb * oof_xgb + w_nli * nli_raw + w_sim * sim_raw
            p = (blended >= 0.5).astype(int)
            f1 = f1_score(y, p, average='macro')
            if f1 > best_f1:
                best_f1 = f1
                best_w = (w_lr, w_xgb, w_nli, w_sim)

w_lr, w_xgb, w_nli, w_sim = best_w
print(f"Optimal weights: LR={w_lr:.2f}, XGB={w_xgb:.2f}, NLI={w_nli:.2f}, SIM={w_sim:.2f}")
print(f"Blended OOF Macro F1: {best_f1:.4f}")

final_blend = w_lr * oof_lr + w_xgb * oof_xgb + w_nli * nli_raw + w_sim * sim_raw
final_p = (final_blend >= 0.5).astype(int)
print(f"Blend distribution: 0={(final_p==0).sum()}, 1={(final_p==1).sum()}")
print(confusion_matrix(y, final_p))
print(classification_report(y, final_p, target_names=['hallucinated(0)', 'faithful(1)']))

## 8. Threshold Optimization on OOF

In [ ]:
# Fine-tune threshold on OOF blend scores
best_t, best_f1_t = 0.5, best_f1
for t in np.arange(0.3, 0.7, 0.01):
    p = (final_blend >= t).astype(int)
    f1 = f1_score(y, p, average='macro')
    if f1 > best_f1_t:
        best_f1_t = f1
        best_t = t

print(f"Optimal threshold: {best_t:.2f}")
print(f"Best OOF Macro F1: {best_f1_t:.4f}")
final_p_opt = (final_blend >= best_t).astype(int)
print(confusion_matrix(y, final_p_opt))

## 9. Train Full Models & Predict Test

In [ ]:
test_df = pd.read_csv(TEST_PATH)
print(f"Test: {len(test_df)}")

# Clean test data
for col in ['context', 'prompt_bn', 'response_bn']:
    test_df[col] = test_df[col].apply(clean)
test_df['has_context'] = test_df['context'].str.len() > 0

In [ ]:
# NLI scores for test
print("Computing test NLI scores...")
test_nli_entail, test_nli_contra = get_nli_scores(test_df)
test_df['nli_entail'] = test_nli_entail
test_df['nli_contra'] = test_nli_contra

# Embedding similarity for test
print("Computing test embedding similarity...")
test_ctx_texts = test_df['context'].where(test_df['context'] != '', test_df['prompt_bn']).tolist()
test_resp_texts = test_df['response_bn'].tolist()
test_prompt_texts = test_df['prompt_bn'].tolist()

test_sim_ctx_resp, test_emb_ctx, test_emb_resp = cosine_sim_matrix(test_ctx_texts, test_resp_texts)
test_sim_prompt_resp, _, _ = cosine_sim_matrix(test_prompt_texts, test_resp_texts)

test_df['sim_ctx_resp'] = test_sim_ctx_resp
test_df['sim_prompt_resp'] = test_sim_prompt_resp

print(f"Test NLI entail mean: {test_nli_entail.mean():.3f}")
print(f"Test sim_ctx_resp mean: {test_sim_ctx_resp.mean():.3f}")

In [ ]:
# Build test features
hc_test = build_features(test_df)

X_test = np.column_stack([
    hc_test.values,
    test_df['nli_entail'].values.reshape(-1, 1),
    test_df['nli_contra'].values.reshape(-1, 1),
    test_df['sim_ctx_resp'].values.reshape(-1, 1),
    test_df['sim_prompt_resp'].values.reshape(-1, 1),
])

print(f"Test features: {X_test.shape}")

# Retrain LR on full data
lr_full = SKPipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced',
        random_state=RANDOM_STATE
    )),
])
lr_full.fit(X_full, y)
lr_test_prob = lr_full.predict_proba(X_test)[:, 1]

# Retrain XGB on full data
xgb_full = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
    scale_pos_weight=n0/n1, random_state=RANDOM_STATE,
    eval_metric='logloss', use_label_encoder=False, verbosity=0
)
xgb_full.fit(X_full, y)
xgb_test_prob = xgb_full.predict_proba(X_test)[:, 1]

print(f"LR test prob mean: {lr_test_prob.mean():.3f}")
print(f"XGB test prob mean: {xgb_test_prob.mean():.3f}")

In [ ]:
# Final blend on test
test_blend = (w_lr * lr_test_prob + w_xgb * xgb_test_prob +
              w_nli * test_nli_entail + w_sim * test_sim_ctx_resp)

test_preds = (test_blend >= best_t).astype(int)

n0_pred = (test_preds == 0).sum()
n1_pred = (test_preds == 1).sum()
print(f"Test distribution: 0={n0_pred} ({n0_pred/len(test_preds)*100:.1f}%), 1={n1_pred} ({n1_pred/len(test_preds)*100:.1f}%)")
print(f"Blend score range: [{test_blend.min():.3f}, {test_blend.max():.3f}]")

## 10. Save Submission

In [ ]:
sub = pd.DataFrame({'id': test_df['id'], 'label': test_preds})
os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved: {SUBMISSION_PATH}")
print(f"0: {n0_pred}, 1: {n1_pred}")
sub.head(10)

## Summary

v4.0 improvements over v2.0/v3.0:
- **No fine-tuning** — avoids overfitting on 299 samples
- **Richer features** — 20+ hand-crafted features capture token overlap, number matching, length ratios
- **Dual NLI signals** — entailment + contradiction scores
- **Embedding similarity** — multilingual sentence-transformers for semantic similarity
- **Proper OOF** — 5-fold CV with grid-searched blend weights and threshold